In [1]:
# [LOCAL PATCH] google.colab removed
import pandas as pd
# [LOCAL PATCH] drive.mount removed


In [2]:
#Detects whether the separator in a file is ';' or ',' based on the first line.
def detect_separator(path, max_lines=1):
    with open(path, 'r', encoding='utf-8') as file:
        line = file.readline()
        return ';' if line.count(';') > line.count(',') else ','

In [3]:
#Loads multiple datasets from CSV files located in the given base path.
def load_dataset(base_path):
    files = {
        "noDependency": "noDependency.csv",
        "noModularity": "noModularity.csv",
        "init": "init.csv",
        "initDesignate": "initTypeMetrics.csv",
        "noModularityDesignate": "noModularityTypeMetrics.csv",
        "noDependencyDesignate": "noDependencyTypeMetrics.csv"
    }

    datasets = {}
    for name, filename in files.items():
        path = base_path + filename
        df = pd.read_csv(path, sep=None, engine="python")
        df.columns = df.columns.str.strip()
        datasets[name] = df

    return datasets


In [4]:
#Calculates the average SumCyclomatic for the specified kind and returns values for noDependency, noModularity, and init datasets.
def calculate_sum_cyclomatic(df_noDep, df_noMod, df_init, kind="Package"):
    df_noDep_filtered = df_noDep[df_noDep["Kind"] == kind]
    df_noMod_filtered = df_noMod[df_noMod["Kind"] == kind]
    df_init_filtered = df_init[df_init["Kind"] == kind]

    avg_noDep = df_noDep_filtered["SumCyclomatic"].mean()
    avg_noMod = df_noMod_filtered["SumCyclomatic"].mean()
    avg_init = df_init_filtered["SumCyclomatic"].mean()

    diff_noDep = avg_noDep - avg_init
    diff_noMod = avg_noMod - avg_init

    return avg_noDep, avg_noMod, avg_init


In [5]:
#Calculates the average hierarchical cohesion (PercentLackOfCohesion) for classes across packages in noDependency, noModularity, and init datasets.
def calculate_avg_hierarchical_cohesion(df_noDep, df_noMod, df_init):
    def average(df):
        df_class = df[df["Kind"].str.contains("Class", case=False, na=False)].copy()
        df_class = df_class.dropna(subset=["PercentLackOfCohesion"])
        if df_class.empty:
            return None
        df_class["Package"] = df_class["Name"].apply(lambda x: ".".join(x.split(".")[:-1]))
        avg_per_package = df_class.groupby("Package")["PercentLackOfCohesion"].mean()
        return avg_per_package.mean()

    avg_noDep = average(df_noDep)
    avg_noMod = average(df_noMod)
    avg_init = average(df_init)

    diff_noDep = avg_noDep - avg_init
    diff_noMod = avg_noMod - avg_init

    return avg_noDep, avg_noMod, avg_init


In [6]:
#Calculates the average sum of class couplings (CountClassCoupled) across packages in noDependency, noModularity, and init datasets.
def calculate_avg_sum_coupling(df_noDep, df_noMod, df_init):
    def average_sums(df):
        df_class = df[df["Kind"].str.contains("Class", case=False, na=False)].copy()
        df_class = df_class.dropna(subset=["CountClassCoupled"])
        if df_class.empty:
            return None
        df_class["Package"] = df_class["Name"].apply(lambda x: ".".join(x.split(".")[:-1]))
        sum_per_package = df_class.groupby("Package")["CountClassCoupled"].sum()
        return sum_per_package.mean()

    avg_noDep = average_sums(df_noDep)
    avg_noMod = average_sums(df_noMod)
    avg_init = average_sums(df_init)

    diff_noDep = avg_noDep - avg_init
    diff_noMod = avg_noMod - avg_init

    return avg_noDep, avg_noMod, avg_init


In [7]:
#Prints quality metrics with differences compared to the init dataset.
def print_quality(metric_name, val_mod, val_dep, val_init):
    delta_mod = round(val_mod - val_init, 2)
    delta_dep = round(val_dep - val_init, 2)
    print(f"{metric_name}")
    print(f"  noModularity: {val_mod:.2f} | Δ vs init: {delta_mod}")
    print(f"  NoDependency: {val_dep:.2f} | Δ vs init: {delta_dep}")
    print(f"  Init: {val_init:.2f}")


In [8]:
#Calculates the average class cohesion (PercentLackOfCohesion) for noDependency, noModularity, and init datasets.
def calculate_avg_class_cohesion(df_noDep, df_noMod, df_init):
    def average(df):
        df_class = df[df["Kind"].str.contains("Class", case=False, na=False)].copy()
        df_class = df_class.dropna(subset=["PercentLackOfCohesion"])
        if df_class.empty:
            return None
        return df_class["PercentLackOfCohesion"].mean()

    avg_noDep = average(df_noDep)
    avg_noMod = average(df_noMod)
    avg_init = average(df_init)

    return avg_noDep, avg_noMod, avg_init



In [9]:
#Calculates the average class coupling (CountClassCoupled) for noDependency, noModularity, and init datasets.
def calculate_avg_class_coupling(df_noDep, df_noMod, df_init):
    def average(df):
        df_class = df[df["Kind"].str.contains("Class", case=False, na=False)].copy()
        df_class = df_class.dropna(subset=["CountClassCoupled"])
        if df_class.empty:
            return None
        return df_class["CountClassCoupled"].mean()

    avg_noDep = average(df_noDep)
    avg_noMod = average(df_noMod)
    avg_init = average(df_init)

    return avg_noDep, avg_noMod, avg_init



In [10]:
#Calculates the average class cyclomatic complexity (SumCyclomatic) for noDependency, noModularity, and init datasets.
def calculate_avg_class_cyclomatic(df_noDep, df_noMod, df_init):
    def average(df):
        df_class = df[df["Kind"].str.contains("Class", case=False, na=False)].copy()
        df_class = df_class.dropna(subset=["SumCyclomatic"])
        if df_class.empty:
            return None
        return df_class["SumCyclomatic"].mean()

    avg_noDep = average(df_noDep)
    avg_noMod = average(df_noMod)
    avg_init = average(df_init)

    return avg_noDep, avg_noMod, avg_init



In [11]:
#Calculates the averages of NC, DIT, FANIN, FANOUT metrics, and LCOM (excluding -1 values if present).
def calculate_avg_metrics(df):
    metric_columns = ["NC", "DIT", "FANIN", "FANOUT"]
    averages = df[metric_columns].mean(numeric_only=True).to_dict()

    if "LCOM" in df.columns:
        valid_lcom = df["LCOM"][df["LCOM"] != -1]
        avg_lcom = valid_lcom.mean()
        averages["LCOM"] = avg_lcom

    return averages

In [12]:
#Calculates project-level metrics by aggregating per package (sum for NC, FANIN, FANOUT and mean for DIT, LCOM).
def calculate_project_metrics(df):
    summable_metrics = ['NC', 'FANIN', 'FANOUT']
    average_metrics = ['DIT', 'LCOM']

    #Ensure numeric type
    for col in summable_metrics + average_metrics:
        df[col] = df[col].astype(float)

    #Aggregation per package
    agg_funcs = {m: 'sum' for m in summable_metrics}
    agg_funcs.update({m: 'mean' for m in average_metrics})

    df_per_package = df.groupby('Package Name', as_index=False).agg(agg_funcs)

    #Calculate average across packages for each metric
    project_values = df_per_package.mean(numeric_only=True).to_dict()

    return project_values

In [13]:
#List of project names
names = ['jackrabbit', 'active', 'Compress', 'ApacheRoller', 'openMrs', 'jeromq', 'oozie']

#Results containers
results = []
results_designite = []

In [14]:
#Iterates over all projects, loads datasets, computes metrics, prints summaries, and collects results.
for project in names:
  print("----------------------------")
  print("Project quality: " + project)

  base_path = r"C:\Users\JAY\Downloads\replication_package\replication_package\quality\data\\" + project + "\\"
  datasets = load_dataset(base_path)

  df_noDep = datasets["noDependency"]
  df_noMod = datasets["noModularity"]
  df_init = datasets["init"]
  df_init_designate = datasets["initDesignate"]
  df_noMod_designate = datasets["noModularityDesignate"]
  df_noDep_designate = datasets["noDependencyDesignate"]

  #PACKAGE-LEVEL METRICS
  cyclomatic_noDep, cyclomatic_noMod, cyclomatic_init = calculate_sum_cyclomatic(df_noDep, df_noMod, df_init, kind="Package")
  cohesion_noDep, cohesion_noMod, cohesion_init = calculate_avg_hierarchical_cohesion(df_noDep, df_noMod, df_init)
  coupling_noDep, coupling_noMod, coupling_init = calculate_avg_sum_coupling(df_noDep, df_noMod, df_init)

  #CLASS-LEVEL METRICS
  cyclomatic_cls_noDep, cyclomatic_cls_noMod, cyclomatic_cls_init = calculate_avg_class_cyclomatic(df_noDep, df_noMod, df_init)
  cohesion_cls_noDep, cohesion_cls_noMod, cohesion_cls_init = calculate_avg_class_cohesion(df_noDep, df_noMod, df_init)
  coupling_cls_noDep, coupling_cls_noMod, coupling_cls_init = calculate_avg_class_coupling(df_noDep, df_noMod, df_init)

  #DESIGNITE METRICS (averages for NC, DIT, FANIN, FANOUT, LCOM with LCOM ≠ -1)
  avgs_init = calculate_avg_metrics(df_init_designate)
  avgs_noMod = calculate_avg_metrics(df_noMod_designate)
  avgs_noDep = calculate_avg_metrics(df_noDep_designate)

  #PACKAGE-LEVEL DESIGNITE METRICS
  print(df_init_designate.columns)
  pkg_avg_init = calculate_project_metrics(df_init_designate)
  pkg_avg_noMod = calculate_project_metrics(df_noMod_designate)
  pkg_avg_noDep = calculate_project_metrics(df_noDep_designate)

  print()
  #PRINT PACKAGE-LEVEL METRICS
  print_quality("Cyclomatic (Package)", cyclomatic_noMod, cyclomatic_noDep, cyclomatic_init)
  print()
  print_quality("Cohesion (Package)", cohesion_noMod, cohesion_noDep, cohesion_init)
  print()
  print_quality("Coupling (Package)", coupling_noMod, coupling_noDep, coupling_init)

  print()
  #PRINT CLASS-LEVEL METRICS
  print_quality("Cyclomatic (Class)", cyclomatic_cls_noMod, cyclomatic_cls_noDep, cyclomatic_cls_init)
  print()
  print_quality("Cohesion (Class)", cohesion_cls_noMod, cohesion_cls_noDep, cohesion_cls_init)
  print()
  print_quality("Coupling (Class)", coupling_cls_noMod, coupling_cls_noDep, coupling_cls_init)
  print()

  print("--- Designite metrics ---")
  for metric in ["NC", "DIT", "FANIN", "FANOUT", "LCOM"]:
    print_quality(f"{metric}", avgs_noMod[metric], avgs_noDep[metric], avgs_init[metric])
  print()

  print("--- Designite metrics (Package) ---")
  for metric in ["NC", "DIT", "FANIN", "FANOUT", "LCOM"]:
    print_quality(f"{metric}", pkg_avg_noMod[metric], pkg_avg_noDep[metric], pkg_avg_init[metric])
  print()

  #SAVE RESULTS (overall)
  results.append({
      "project": project,
      "version": "No modular smells",
      "Cyclomatic (Package)": (round(cyclomatic_noMod, 4), round(cyclomatic_noMod - cyclomatic_init, 4)),
      "Cohesion (Package)": (round(cohesion_noMod, 4), round(cohesion_noMod - cohesion_init, 4)),
      "Coupling (Package)": (round(coupling_noMod, 4), round(coupling_noMod - coupling_init, 4)),
      "Cyclomatic (Class)": (round(cyclomatic_cls_noMod, 4), round(cyclomatic_cls_noMod - cyclomatic_cls_init, 4)),
      "Cohesion (Class)": (round(cohesion_cls_noMod, 4), round(cohesion_cls_noMod - cohesion_cls_init, 4)),
      "Coupling (Class)": (round(coupling_cls_noMod, 4), round(coupling_cls_noMod - coupling_cls_init, 4))
  })

  results.append({
      "project": project,
      "version": "No dependency smells",
      "Cyclomatic (Package)": (round(cyclomatic_noDep, 4), round(cyclomatic_noDep - cyclomatic_init, 4)),
      "Cohesion (Package)": (round(cohesion_noDep, 4), round(cohesion_noDep - cohesion_init, 4)),
      "Coupling (Package)": (round(coupling_noDep, 4), round(coupling_noDep - coupling_init, 4)),
      "Cyclomatic (Class)": (round(cyclomatic_cls_noDep, 4), round(cyclomatic_cls_noDep - cyclomatic_cls_init, 4)),
      "Cohesion (Class)": (round(cohesion_cls_noDep, 4), round(cohesion_cls_noDep - cohesion_cls_init, 4)),
      "Coupling (Class)": (round(coupling_cls_noDep, 4), round(coupling_cls_noDep - coupling_cls_init, 4))
  })

  #SAVE RESULTS (Designite)
  results_designite.append({
      "project": project,
      "version": "No modular smells",
      "NC": (round(avgs_noMod["NC"], 4), round(avgs_noMod["NC"] - avgs_init["NC"], 4)),
      "DIT": (round(avgs_noMod["DIT"], 4), round(avgs_noMod["DIT"] - avgs_init["DIT"], 4)),
      "FANIN": (round(avgs_noMod["FANIN"], 4), round(avgs_noMod["FANIN"] - avgs_init["FANIN"], 4)),
      "FANOUT": (round(avgs_noMod["FANOUT"], 4), round(avgs_noMod["FANOUT"] - avgs_init["FANOUT"], 4)),
      "LCOM": (round(avgs_noMod["LCOM"], 4), round(avgs_noMod["LCOM"] - avgs_init["LCOM"], 4)),
      "NC (Package)": (round(pkg_avg_noMod["NC"], 4), round(pkg_avg_noMod["NC"] - pkg_avg_init["NC"], 4)),
      "DIT (Package)": (round(pkg_avg_noMod["DIT"], 4), round(pkg_avg_noMod["DIT"] - pkg_avg_init["DIT"], 4)),
      "FANIN (Package)": (round(pkg_avg_noMod["FANIN"], 4), round(pkg_avg_noMod["FANIN"] - pkg_avg_init["FANIN"], 4)),
      "FANOUT (Package)": (round(pkg_avg_noMod["FANOUT"], 4), round(pkg_avg_noMod["FANOUT"] - pkg_avg_init["FANOUT"], 4)),
      "LCOM (Package)": (round(pkg_avg_noMod["LCOM"], 4), round(pkg_avg_noMod["LCOM"] - pkg_avg_init["LCOM"], 4)),
  })

  results_designite.append({
      "project": project,
      "version": "No dependency smells",
      "NC": (round(avgs_noDep["NC"], 4), round(avgs_noDep["NC"] - avgs_init["NC"], 4)),
      "DIT": (round(avgs_noDep["DIT"], 4), round(avgs_noDep["DIT"] - avgs_init["DIT"], 4)),
      "FANIN": (round(avgs_noDep["FANIN"], 4), round(avgs_noDep["FANIN"] - avgs_init["FANIN"], 4)),
      "FANOUT": (round(avgs_noDep["FANOUT"], 4), round(avgs_noDep["FANOUT"] - avgs_init["FANOUT"], 4)),
      "LCOM": (round(avgs_noDep["LCOM"], 4), round(avgs_noDep["LCOM"] - avgs_init["LCOM"], 4)),
      "NC (Package)": (round(pkg_avg_noDep["NC"], 4), round(pkg_avg_noDep["NC"] - pkg_avg_init["NC"], 4)),
      "DIT (Package)": (round(pkg_avg_noDep["DIT"], 4), round(pkg_avg_noDep["DIT"] - pkg_avg_init["DIT"], 4)),
      "FANIN (Package)": (round(pkg_avg_noDep["FANIN"], 4), round(pkg_avg_noDep["FANIN"] - pkg_avg_init["FANIN"], 4)),
      "FANOUT (Package)": (round(pkg_avg_noDep["FANOUT"], 4), round(pkg_avg_noDep["FANOUT"] - pkg_avg_init["FANOUT"], 4)),
      "LCOM (Package)": (round(pkg_avg_noDep["LCOM"], 4), round(pkg_avg_noDep["LCOM"] - pkg_avg_init["LCOM"], 4)),
  })



----------------------------
Project quality: jackrabbit


Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 168.84 | Δ vs init: -110.03
  NoDependency: 275.19 | Δ vs init: -3.67
  Init: 278.87

Cohesion (Package)
  noModularity: 17.29 | Δ vs init: 2.24
  NoDependency: 15.05 | Δ vs init: 0.01
  Init: 15.04

Coupling (Package)
  noModularity: 32.73 | Δ vs init: -3.59
  NoDependency: 36.24 | Δ vs init: -0.08
  Init: 36.32

Cyclomatic (Class)
  noModularity: 16.39 | Δ vs init: 0.0
  NoDependency: 16.37 | Δ vs init: -0.02
  Init: 16.39

Cohesion (Class)
  noModularity: 27.41 | Δ vs init: 0.02
  NoDependency: 27.36 | Δ vs init: -0.03
  Init: 27.39

Coupling (Class)
  noModularity: 10.61 | Δ vs init: 0.0
  NoDependency: 10.60 | Δ vs init: -0.01
  Init: 10.61

--- Designite metrics ---
NC
  noModularity: 0.34 | Δ vs init: -0.03
  NoDependency: 0.37 | Δ vs init: -0.0
  

Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 148.16 | Δ vs init: -160.84
  NoDependency: 296.97 | Δ vs init: -12.03
  Init: 309.00

Cohesion (Package)
  noModularity: 10.49 | Δ vs init: 3.05
  NoDependency: 7.49 | Δ vs init: 0.06
  Init: 7.43

Coupling (Package)
  noModularity: 28.17 | Δ vs init: -2.65
  NoDependency: 30.74 | Δ vs init: -0.09
  Init: 30.82

Cyclomatic (Class)
  noModularity: 9.49 | Δ vs init: 0.0
  NoDependency: 9.49 | Δ vs init: 0.0
  Init: 9.49

Cohesion (Class)
  noModularity: 22.93 | Δ vs init: -0.0
  NoDependency: 22.94 | Δ vs init: -0.0
  Init: 22.94

Coupling (Class)
  noModularity: 9.33 | Δ vs init: 0.01
  NoDependency: 9.33 | Δ vs init: 0.0
  Init: 9.33

--- Designite metrics ---
NC
  noModularity: 0.42 | Δ vs init: -0.03
  NoDependency: 0.45 | Δ vs init: 0.0
  Init: 0.44
D

Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 173.03 | Δ vs init: -115.33
  NoDependency: 282.11 | Δ vs init: -6.25
  Init: 288.36

Cohesion (Package)
  noModularity: 21.13 | Δ vs init: -3.84
  NoDependency: 24.78 | Δ vs init: -0.19
  Init: 24.97

Coupling (Package)
  noModularity: 47.81 | Δ vs init: -2.34
  NoDependency: 49.75 | Δ vs init: -0.4
  Init: 50.15

Cyclomatic (Class)
  noModularity: 16.99 | Δ vs init: -0.04
  NoDependency: 16.99 | Δ vs init: -0.04
  Init: 17.03

Cohesion (Class)
  noModularity: 27.02 | Δ vs init: -0.06
  NoDependency: 27.04 | Δ vs init: -0.04
  Init: 27.08

Coupling (Class)
  noModularity: 9.40 | Δ vs init: -0.02
  NoDependency: 9.39 | Δ vs init: -0.03
  Init: 9.42

--- Designite metrics ---
NC
  noModularity: 0.26 | Δ vs init: 0.0
  NoDependency: 0.25 | Δ vs init: -0.01


Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 56.80 | Δ vs init: -59.58
  NoDependency: 115.00 | Δ vs init: -1.37
  Init: 116.37

Cohesion (Package)
  noModularity: 40.97 | Δ vs init: 10.31
  NoDependency: 30.30 | Δ vs init: -0.36
  Init: 30.66

Coupling (Package)
  noModularity: 45.62 | Δ vs init: -35.31
  NoDependency: 80.21 | Δ vs init: -0.71
  Init: 80.92

Cyclomatic (Class)
  noModularity: 17.26 | Δ vs init: 0.0
  NoDependency: 17.16 | Δ vs init: -0.09
  Init: 17.25

Cohesion (Class)
  noModularity: 43.03 | Δ vs init: 0.08
  NoDependency: 42.84 | Δ vs init: -0.11
  Init: 42.95

Coupling (Class)
  noModularity: 13.06 | Δ vs init: -0.01
  NoDependency: 13.03 | Δ vs init: -0.04
  Init: 13.07

--- Designite metrics ---
NC
  noModularity: 0.20 | Δ vs init: -0.01
  NoDependency: 0.21 | Δ vs init: -0.0

Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 93.78 | Δ vs init: -132.77
  NoDependency: 214.67 | Δ vs init: -11.88
  Init: 226.55

Cohesion (Package)
  noModularity: 21.93 | Δ vs init: -3.0
  NoDependency: 24.32 | Δ vs init: -0.62
  Init: 24.94

Coupling (Package)
  noModularity: 55.40 | Δ vs init: -39.37
  NoDependency: 92.08 | Δ vs init: -2.69
  Init: 94.77

Cyclomatic (Class)
  noModularity: 17.61 | Δ vs init: 0.0
  NoDependency: 17.50 | Δ vs init: -0.11
  Init: 17.61

Cohesion (Class)
  noModularity: 26.49 | Δ vs init: -0.01
  NoDependency: 26.32 | Δ vs init: -0.18
  Init: 26.50

Coupling (Class)
  noModularity: 12.59 | Δ vs init: 0.0
  NoDependency: 12.55 | Δ vs init: -0.04
  Init: 12.59

--- Designite metrics ---
NC
  noModularity: 0.25 | Δ vs init: -0.07
  NoDependency: 0.30 | Δ vs init: -0.0

Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 61.38 | Δ vs init: -119.64
  NoDependency: 176.69 | Δ vs init: -4.33
  Init: 181.02

Cohesion (Package)
  noModularity: 14.07 | Δ vs init: -0.27
  NoDependency: 13.98 | Δ vs init: -0.36
  Init: 14.34

Coupling (Package)
  noModularity: 26.32 | Δ vs init: -7.65
  NoDependency: 34.59 | Δ vs init: 0.62
  Init: 33.97

Cyclomatic (Class)
  noModularity: 11.02 | Δ vs init: 0.01
  NoDependency: 10.98 | Δ vs init: -0.03
  Init: 11.01

Cohesion (Class)
  noModularity: 16.17 | Δ vs init: 0.0
  NoDependency: 16.13 | Δ vs init: -0.05
  Init: 16.17

Coupling (Class)
  noModularity: 8.31 | Δ vs init: -0.01
  NoDependency: 8.45 | Δ vs init: 0.13
  Init: 8.33

--- Designite metrics ---
NC
  noModularity: 0.09 | Δ vs init: 0.0
  NoDependency: 0.10 | Δ vs init: 0.01
  Init

Index(['Project Name', 'Package Name', 'Type Name', 'NOF', 'NOPF', 'NOM',
       'NOPM', 'LOC', 'WMC', 'NC', 'DIT', 'LCOM', 'FANIN', 'FANOUT',
       'File path', 'Line no'],
      dtype='object')

Cyclomatic (Package)
  noModularity: 153.88 | Δ vs init: -173.57
  NoDependency: 323.43 | Δ vs init: -4.02
  Init: 327.45

Cohesion (Package)
  noModularity: 8.39 | Δ vs init: 1.24
  NoDependency: 7.15 | Δ vs init: -0.0
  Init: 7.15

Coupling (Package)
  noModularity: 33.47 | Δ vs init: -2.13
  NoDependency: 35.17 | Δ vs init: -0.43
  Init: 35.60

Cyclomatic (Class)
  noModularity: 10.71 | Δ vs init: 0.0
  NoDependency: 10.69 | Δ vs init: -0.02
  Init: 10.71

Cohesion (Class)
  noModularity: 23.09 | Δ vs init: -0.0
  NoDependency: 23.06 | Δ vs init: -0.03
  Init: 23.09

Coupling (Class)
  noModularity: 12.34 | Δ vs init: 0.02
  NoDependency: 12.14 | Δ vs init: -0.18
  Init: 12.32

--- Designite metrics ---
NC
  noModularity: 0.29 | Δ vs init: -0.01
  NoDependency: 0.30 | Δ vs init: -0.0
  In

In [15]:
#Create a DataFrame from the collected results
df_results = pd.DataFrame(results)
df_results


,project,version,Cyclomatic (Package),Cohesion (Package),Coupling (Package),Cyclomatic (Class),Cohesion (Class),Coupling (Class)
0,jackrabbit,No modular smells,"(168.8405, -110.0255)","(17.2854, 2.2436)","(32.7344, -3.5872)","(16.3879, 0.0013)","(27.411, 0.0168)","(10.6108, 0.0003)"
1,jackrabbit,No dependency smells,"(275.1938, -3.6722)","(15.0541, 0.0123)","(36.2419, -0.0797)","(16.3657, -0.021)","(27.3615, -0.0328)","(10.6018, -0.0087)"
2,active,No modular smells,"(148.1577, -160.8378)","(10.4868, 3.0526)","(28.1738, -2.651)","(9.4889, 0.0)","(22.9349, -0.0032)","(9.3315, 0.0051)"
3,active,No dependency smells,"(296.9654, -12.0301)","(7.4913, 0.0571)","(30.7386, -0.0862)","(9.4893, 0.0005)","(22.9374, -0.0007)","(9.3271, 0.0007)"
4,Compress,No modular smells,"(173.0267, -115.3289)","(21.1276, -3.8397)","(47.8054, -2.3425)","(16.9868, -0.0436)","(27.0198, -0.0596)","(9.3971, -0.0222)"
5,Compress,No dependency smells,"(282.1087, -6.2469)","(24.7773, -0.19)","(49.7483, -0.3996)","(16.9868, -0.0436)","(27.0356, -0.0437)","(9.3852, -0.0341)"
6,ApacheRoller,No modular smells,"(56.7952, -59.5752)","(40.9674, 10.3054)","(45.6154, -35.3051)","(17.2569, 0.0037)","(43.0349, 0.0844)","(13.0569, -0.0092)"
7,ApacheRoller,No dependency smells,"(115.0, -1.3704)","(30.3012, -0.3608)","(80.2135, -0.707)","(17.1642, -0.089)","(42.8431, -0.1074)","(13.0274, -0.0387)"
8,openMrs,No modular smells,"(93.7814, -132.7692)","(21.9304, -3.0046)","(55.4038, -39.3659)","(17.6119, 0.0)","(26.493, -0.0061)","(12.5918, 0.0)"
9,openMrs,No dependency smells,"(214.6702, -11.8803)","(24.3178, -0.6172)","(92.0764, -2.6933)","(17.5035, -0.1084)","(26.3151, -0.184)","(12.5486, -0.0432)"


In [16]:
#Create a DataFrame from the collected Designite results
df_results_designite = pd.DataFrame(results_designite)
df_results_designite

,project,version,NC,DIT,FANIN,FANOUT,LCOM,NC (Package),DIT (Package),FANIN (Package),FANOUT (Package),LCOM (Package)
0,jackrabbit,No modular smells,"(0.3422, -0.0263)","(0.4103, -0.054)","(2.9062, -0.0138)","(2.9062, -0.0138)","(0.2279, -0.0017)","(3.3128, -2.6267)","(0.2413, -0.0735)","(28.1369, -18.9375)","(28.1369, -18.9375)","(-0.2502, 0.0477)"
1,jackrabbit,No dependency smells,"(0.3667, -0.0017)","(0.4628, -0.0015)","(2.9022, -0.0179)","(2.9022, -0.0179)","(0.2297, 0.0002)","(5.8486, -0.0909)","(0.3151, 0.0002)","(46.289, -0.7854)","(46.289, -0.7854)","(-0.3076, -0.0097)"
2,active,No modular smells,"(0.4151, -0.0297)","(0.6924, -0.0535)","(2.7572, 0.0136)","(2.7572, 0.0136)","(0.2349, -0.0008)","(4.6102, -5.9517)","(0.2711, -0.1212)","(30.6236, -34.5288)","(30.6236, -34.5288)","(-0.1838, 0.0107)"
3,active,No dependency smells,"(0.4453, 0.0006)","(0.7452, -0.0008)","(2.7394, -0.0042)","(2.7394, -0.0042)","(0.2358, 0.0001)","(10.1735, -0.3884)","(0.3808, -0.0115)","(62.5799, -2.5725)","(62.5799, -2.5725)","(-0.211, -0.0165)"
4,Compress,No modular smells,"(0.263, 0.0006)","(0.335, -0.082)","(2.1113, 0.0998)","(2.1113, 0.0998)","(0.262, -0.0091)","(3.1045, -2.4901)","(0.1972, -0.0328)","(24.9254, -17.9665)","(24.9254, -17.9665)","(-0.3948, 0.0116)"
5,Compress,No dependency smells,"(0.2478, -0.0146)","(0.3648, -0.0522)","(1.9547, -0.0567)","(1.9547, -0.0567)","(0.2712, 0.0002)","(5.1842, -0.4104)","(0.1937, -0.0364)","(40.8947, -1.9972)","(40.8947, -1.9972)","(-0.3951, 0.0113)"
6,ApacheRoller,No modular smells,"(0.1971, -0.0129)","(0.2601, -0.0194)","(3.7367, 0.2149)","(3.7367, 0.2149)","(0.2715, -0.0051)","(0.7974, -1.0598)","(0.1444, -0.0839)","(15.1176, -16.0252)","(15.1176, -16.0252)","(-0.0571, 0.0369)"
7,ApacheRoller,No dependency smells,"(0.2051, -0.0049)","(0.2724, -0.007)","(3.4888, -0.033)","(3.4888, -0.033)","(0.2759, -0.0006)","(1.8028, -0.0543)","(0.2136, -0.0147)","(30.662, -0.4809)","(30.662, -0.4809)","(-0.0853, 0.0087)"
8,openMrs,No modular smells,"(0.2472, -0.071)","(0.5518, -0.0941)","(2.9482, -0.0064)","(2.9482, -0.0064)","(0.3565, -0.0025)","(1.4904, -3.3168)","(0.0764, -0.0946)","(17.774, -26.8645)","(17.774, -26.8645)","(-0.2436, -0.0072)"
9,openMrs,No dependency smells,"(0.3028, -0.0154)","(0.604, -0.042)","(2.9146, -0.0399)","(2.9146, -0.0399)","(0.3592, 0.0003)","(4.4535, -0.3537)","(0.1602, -0.0108)","(42.8721, -1.7665)","(42.8721, -1.7665)","(-0.2284, 0.008)"


In [17]:
print(df_results.columns.tolist())


['project', 'version', 'Cyclomatic (Package)', 'Cohesion (Package)', 'Coupling (Package)', 'Cyclomatic (Class)', 'Cohesion (Class)', 'Coupling (Class)']


In [18]:
from openpyxl import load_workbook

#Load the existing Excel file
path_excel = r"C:\Users\JAY\Downloads\replication_package\replication_package\quality\QualityResults.xlsx"
wb = load_workbook(path_excel)
ws = wb["Foglio1"]

#Updated column mapping
columns_map = {
    "Cyclomatic (Package)": 4,
    "Cohesion (Package)": 5,
    "Coupling (Package)": 6,
    "Cyclomatic (Class)": 7,
    "Cohesion (Class)": 8,
    "Coupling (Class)": 9
}

#Starting row in Excel = 7
start_row = 7

#Write values into the cells for all metrics
for i, (_, row) in enumerate(df_results.iterrows()):
    excel_row = start_row + i
    for metric, col in columns_map.items():
        val, delta = row[metric]
        ws.cell(row=excel_row, column=col).value = f"{val:.4f} ({delta:+.4f})"

#=== APPEND DESIGNITE RESULTS AT THE END OF "Foglio1" ===
#Skip 2 empty rows and continue after df_results
offset = len(df_results) + start_row + 6
ws.cell(row=offset - 1, column=2).value = "Designite Metrics"
ws.cell(row=offset - 1, column=5).value = "NC"
ws.cell(row=offset - 1, column=6).value = "DIT"
ws.cell(row=offset - 1, column=7).value = "FANIN"
ws.cell(row=offset - 1, column=8).value = "FANOUT"
ws.cell(row=offset - 1, column=9).value = "LCOM"
ws.cell(row=offset - 1, column=11).value = "Package"
ws.cell(row=offset - 1, column=12).value = "NC"
ws.cell(row=offset - 1, column=13).value = "DIT"
ws.cell(row=offset - 1, column=14).value = "FANIN"
ws.cell(row=offset - 1, column=15).value = "FANOUT"
ws.cell(row=offset - 1, column=16).value = "LCOM"

for i, row in enumerate(results_designite):
    excel_row = offset + i
    ws.cell(row=excel_row, column=2).value = row["project"]
    ws.cell(row=excel_row, column=3).value = row["version"]
    ws.cell(row=excel_row, column=5).value = f"{row['NC'][0]:.4f} ({row['NC'][1]:+.4f})"
    ws.cell(row=excel_row, column=6).value = f"{row['DIT'][0]:.4f} ({row['DIT'][1]:+.4f})"
    ws.cell(row=excel_row, column=7).value = f"{row['FANIN'][0]:.4f} ({row['FANIN'][1]:+.4f})"
    ws.cell(row=excel_row, column=8).value = f"{row['FANOUT'][0]:.4f} ({row['FANOUT'][1]:+.4f})"
    ws.cell(row=excel_row, column=9).value = f"{row['LCOM'][0]:.4f} ({row['LCOM'][1]:+.4f})"

    ws.cell(row=excel_row, column=12).value = f"{row['NC (Package)'][0]:.4f} ({row['NC (Package)'][1]:+.4f})"
    ws.cell(row=excel_row, column=13).value = f"{row['DIT (Package)'][0]:.4f} ({row['DIT (Package)'][1]:+.4f})"
    ws.cell(row=excel_row, column=14).value = f"{row['FANIN (Package)'][0]:.4f} ({row['FANIN (Package)'][1]:+.4f})"
    ws.cell(row=excel_row, column=15).value = f"{row['FANOUT (Package)'][0]:.4f} ({row['FANOUT (Package)'][1]:+.4f})"
    ws.cell(row=excel_row, column=16).value = f"{row['LCOM (Package)'][0]:.4f} ({row['LCOM (Package)'][1]:+.4f})"

#Save the updated file
wb.save(r"C:\Users\JAY\Downloads\replication_package\replication_package\quality\QualityResults.xlsx")

